In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

from regpy.operators import Operator
from regpy.vecsps import DirectSum

In [ ]:
from FEM_Space import OneDimensionalFEM, LegendreSpace, L2

In [ ]:
fem = OneDimensionalFEM(n_nodes=50,a = -1,b = 1)

leg_dom = LegendreSpace(degree=30, x = fem.global_quad_pts, w = fem.global_quad_weights)
# leg_dom = LegendreSpace(degree=20, n_nodes=50)

# print(fem.global_quad_pts,leg_dom.x)
# x = fem.global_quad_pts
# print(x/(x[-1]-x[0])*2)

plt.figure()
plt.imshow(leg_dom.basis.T@leg_dom.basis)
plt.colorbar()

In [ ]:
f = np.sin(10*leg_dom.x)
f = -0.5*leg_dom.x+0.5
coeff = leg_dom.pts2coeff(f)

plt.figure()
plt.plot(leg_dom.x,f,color="green")
plt.plot(leg_dom.x,leg_dom.coeff2pts(coeff))
plt.figure()
print(leg_dom.ones())
plt.plot(leg_dom.x,leg_dom.coeff2pts(leg_dom.ones()))
one_c = np.ones_like(leg_dom.x)
print(leg_dom.pts2coeff(one_c),leg_dom.pts2coeff(one_c)[0] )
plt.plot(leg_dom.x,leg_dom.coeff2pts(leg_dom.pts2coeff(one_c)))

In [ ]:
plt.plot(leg_dom.x,leg_dom.coeff2pts(leg_dom.ones()), label="Constant Function")

In [ ]:
# leg_dom = LegendreSpace(degree=6,n_nodes= 101)

print(sp.csc_matrix((2/leg_dom.n_nodes)**2 * leg_dom.basis.T@leg_dom.basis).shape, leg_dom.shape)

l2 = L2(leg_dom)

x = leg_dom.ones()
# x = leg_dom.zeros()
# x[2] = 1

print(l2.norm(x))
coeff = leg_dom.pts2coeff(leg_dom.coeff2pts(x))
print(l2.norm(coeff))

print(l2.gram_inv(l2.gram(x)),x)


In [ ]:
legdomain = LegendreSpace(degree=30, n_nodes=1000)

# one = legdomain.zeros()
# one[0] = 1
# one[-1] = 1
# plt.plot(legdomain.x, legdomain.coeff2pts(one), label="Legendre Series")
plt.plot(legdomain.x,legdomain.coeff2pts(legdomain.ones()), label="Constant Function")

l2 = L2(legdomain)

x = legdomain.ones()
# x = leg_dom.zeros()
# x[2] = 1

print(l2.norm(x))
coeff = legdomain.pts2coeff(legdomain.coeff2pts(x))
print(l2.norm(coeff))

z = np.sin(-10*legdomain.x**2)
# z = -5*legdomain.x
z_coeffs = legdomain.pts2coeff(z)
plt.figure()
plt.plot(legdomain.x, z, label="Normal Function")
plt.plot(legdomain.x, legdomain.coeff2pts(z_coeffs), label="Approximation Function")
plt.legend()

In [ ]:
domain = OneDimensionalFEM(p=1, n_nodes=51, a=-1, b=1)
import matplotlib.pyplot as plt
plt.spy(domain.M)
np.allclose(domain.M.todense(), domain.M.todense().T)  # Check if M is symmetric
eigvals = sp.linalg.eigsh(domain.M)[0]
print(eigvals)
ones_quad = np.exp(-domain.global_quad_pts**2)  # Example function to project
ones_quad = np.ones_like(domain.global_quad_pts)  # Example function to project
ones = domain.get_fem_from_pts(ones_quad)
print(ones.shape, domain.M.shape)
print(sum(ones * (domain.M @ ones)))
print(domain.nodes[-1]-domain.nodes[0])

In [ ]:
class FokkerPlanckOp(Operator):
    def __init__(self, sigma, delta_t, fem, legendre_domain, vector_space, B_diff):
        self.fem = fem
        self.sigma = sigma
        self.delta_t = delta_t
        self.legendre_domain = legendre_domain
        self.vector_space = vector_space
        self.B_diff = B_diff
        super().__init__(domain=self.vector_space, codomain=self.fem)

    @staticmethod
    def create_shared(sigma=0.5, delta_t=0.01, p_fem=3, p_legendre=31, n_nodes=51, a=-1, b=1):
        fem = OneDimensionalFEM(p=p_fem, n_nodes=n_nodes, a=a, b=b)
        legendre_domain = LegendreSpace(degree=p_legendre, x = fem.global_quad_pts, w = fem.global_quad_weights)
        vector_space = DirectSum(fem, legendre_domain)
        B_diff = 0.5 * sigma**2 * (fem.DOF2pts_der.T @ sp.diags(fem.global_quad_weights)) @ fem.DOF2pts_der
        return {
            "sigma" : sigma,
            "delta_t" : delta_t,
            "fem": fem,
            "legendre_domain": legendre_domain,
            "vector_space": vector_space,
            "B_diff": B_diff
        }

    def _eval(self, x, differentiate = False):
        """
        Evaluate one time step the Fokker-Planck operator on the input x.
        """
        u, drift = self.domain.split(x)
        drift_pts = self.fem.global_quad_weights * self.legendre_domain.coeff2pts(drift)
        # system  matrix -0.5*sigma^2*u'' + (drift*u)' 
        B = self.B_diff - self.fem.DOF2pts_der.T @ sp.diags(drift_pts) @ self.fem.DOF2pts 
      
        # time step
        system_matrix = (self.fem.M + self.delta_t * B).tocsc()
        self.system_inv = sp.linalg.splu(system_matrix)
        u_next = self.system_inv.solve(self.fem.M @ u)
        if differentiate:
            self.system_matrix = system_matrix
            self.u_j = self.fem.DOF2pts@u_next
        return u_next

    def _derivative(self, h):
        h_u, h_drift = self.domain.split(h)
        h_drift_pts = self.fem.global_quad_weights * self.legendre_domain.coeff2pts(h_drift)
        rhs = self.fem.M @ h_u + (self.delta_t*sp.eye(self.fem.nr_dofs)) @ (self.fem.DOF2pts_der.T @ (h_drift_pts * self.u_j))
        h_u = self.system_inv.solve(rhs)
        return h_u
    
    def _adjoint(self, y):
        y_u = y
        y_step = self.system_inv.solve(y_u,'T')
        
        y_drift_pts = self.u_j * (self.fem.DOF2pts_der @ (self.delta_t*sp.eye(self.fem.nr_dofs)) @ y_step) 
        y_u = self.fem.M.T @ y_step
        return self.domain.join(y_u, self.legendre_domain.pts2coeff(y_drift_pts))

In [ ]:
from regpy.util.operator_tests import test_operator, test_adjoint, test_linearity, test_derivative
shared_data = FokkerPlanckOp.create_shared(sigma=2, delta_t=0.5, p_fem=2, n_nodes=11, a=-1, b=1)
fokker = FokkerPlanckOp(**shared_data)
# test_operator(fokker)
# test_derivative(fokker)
x = fokker.domain.rand()
_, deriv = fokker.linearize(x)
test_linearity(deriv)
test_linearity(deriv.adjoint)
test_adjoint(deriv)

In [ ]:
d = fokker.fem.DOF2pts 
print(d.shape)
m = d@ d.T
print(type(m))
plt.imshow((d.T@d).todense())
plt.colorbar()

In [ ]:
print("Condition Number:", np.linalg.cond(fokker.system_matrix.toarray()))

In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=4, delta_t=1, p_fem=3, p_legendre= 6,  n_nodes=201, a=-5, b=5)
fokker = FokkerPlanckOp(**shared_data)
drift_pts = -fokker.legendre_domain.x
drift = fokker.legendre_domain.pts2coeff(drift_pts)
# drift_pts = -fokker.fem.global_quad_pts
# drift = fokker.fem.get_fem_from_pts(drift_pts)
plt.figure()
# plt.plot(fokker.legendre_domain.x,fokker.fem.DOF2pts @ drift, label="Drift term approximate")
plt.plot(fokker.legendre_domain.x,fokker.legendre_domain.coeff2pts(drift), label="Drift term approximate")
plt.plot(fokker.legendre_domain.x, drift_pts, '--', label="Drift term at quadrature points")
plt.legend()

In [ ]:
print(drift)

In [ ]:



ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)

u_inf_pts = 1 / np.sqrt(2*np.pi) *np.exp(-(fokker.fem.global_quad_pts)**2/2)
# u_inf_pts = np.sqrt(4/2*np.pi) *np.exp(-4*(fokker.fem.global_quad_pts)**2/2)
u_inf = fokker.fem.get_fem_from_pts(u_inf_pts)
scale = ones.T @ fokker.fem.M @ u_inf
u_inf_pts /= scale
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_inf_pts, label="Limit")


u_0_pts = 1 / np.sqrt(0.2*np.pi) *np.exp(-(fokker.fem.global_quad_pts-2)**2/0.2)
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_0_pts, label="Initial condition")
u_0 = fokker.fem.get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)
u_0 /= ones.T @ fokker.fem.M @ u_0
x_plot = fokker.fem.global_quad_pts
plt.plot(x_plot, fokker.fem.DOF2pts @ u_0, label="FEM projection of initial condition")

u_t = np.copy(u_0)
plt.figure()
plt.plot(x_plot, fokker.fem.DOF2pts @ u_t)
Ut = []
for _ in range(500):
    # print("||u_t|| =", ones.T @ fokker.fem.M @ u_t)
    Ut.append(fokker.fem.DOF2pts @ u_t)
    u_t_next = fokker(fokker.domain.join(u_t, drift))
    plt.plot(x_plot[::fokker.fem.p], (fokker.fem.DOF2pts @ u_t_next)[::fokker.fem.p], color = "blue", label="FEM projection")
    u_t = np.copy(u_t_next)
plt.plot(fokker.fem.global_quad_pts, u_inf_pts, color = "orange", label="Limit")
Ut = np.abs(np.array(Ut))
plt.figure()
from matplotlib.colors import LogNorm
plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 500), origin='lower')
# plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 300), origin='lower',norm=LogNorm(vmin=np.min(Ut), vmax=np.max(Ut)))

In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=0.5, delta_t=0.1, p_fem=8, p_legendre=6, n_nodes=201, a=-5, b=5)
fokker = FokkerPlanckOp(**shared_data)


drift_pts = -50*(fokker.legendre_domain.x**3-0.5*fokker.legendre_domain.x) - 0.05
# drift_pts = np.zeros_like(fokker.fem.global_quad_pts)
# drift_pts = - fokker.legendre_domain.x
# drift_pts = np.sin(np.pi * fokker.legendre_domain.x)
drift = fokker.legendre_domain.pts2coeff(drift_pts)
plt.figure()
plt.plot(fokker.legendre_domain.x,fokker.legendre_domain.coeff2pts(drift), label="Drift term approximate")
plt.plot(fokker.legendre_domain.x, drift_pts, '--', label="Drift term at quadrature points")
plt.legend()


shift = 0
u_0_pts = np.zeros_like(fokker.fem.global_quad_pts)
u_0_pts[len(fokker.fem.global_quad_pts) // 2 +shift] = 1.0  # Initial condition: delta function at the center
# u_0_pts = np.exp(-100 * (fokker.fem.global_quad_pts*10+2)**2)
# u_0_pts = 1 / np.sqrt(0.2*np.pi) *np.exp(-(fokker.fem.global_quad_pts-2)**2/0.2)
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_0_pts, label="Initial condition")
u_0 = fokker.fem.get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)
u_0 /= ones.T @ fokker.fem.M @ u_0
x_plot = fokker.fem.global_quad_pts
plt.plot(x_plot, fokker.fem.DOF2pts @ u_0, label="FEM projection of initial condition")


In [ ]:
n = 600

u_t = np.copy(u_0)
plt.figure()
plt.plot(x_plot, fokker.fem.DOF2pts @ u_t)
Ut = []

for _ in range(n):
    # print("||u_t|| =", ones.T @ fokker.fem.M @ u_t)
    Ut.append(fokker.fem.DOF2pts @ u_t)
    u_t_next = fokker(fokker.domain.join(u_t, drift))
    plt.plot(x_plot[::fokker.fem.p], (fokker.fem.DOF2pts @ u_t_next)[::fokker.fem.p], label="FEM projection")
    u_t = np.copy(u_t_next)

Ut = np.abs(np.array(Ut))
plt.figure()
from matplotlib.colors import LogNorm
plt.imshow(Ut, aspect='auto', extent=(-5, 5, 0, n), origin='lower', vmin = 0, vmax= 5)
# plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 300), origin='lower',norm=LogNorm(vmin=np.min(Ut), vmax=np.max(Ut)))
plt.figure()
plt.plot(x_plot, fokker.legendre_domain.coeff2pts(drift), label="FEM projection at final time")


In [ ]:
u_final = fokker.fem.DOF2pts@ u_t

u_final[u_final<0] = 0
plt.plot(x_plot,u_final)


n_sample = 100
sample = np.random.poisson(lam = u_final, size=(n_sample,u_final.size)).sum(axis = 0)/n_sample

plt.plot(x_plot,sample)


In [ ]:
from basic_operator_chain import time_dependent_operator_chain

In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=4, delta_t=0.5, p_fem=3, p_legendre=6, n_nodes=201, a=-5, b=5)
time_steps = [FokkerPlanckOp(**shared_data) for _ in range(500)]

op = time_dependent_operator_chain(time_step_ops=time_steps)

In [ ]:
drift_pts = -op.domain.summands[1].x
drift_pts = -50*(op.domain.summands[1].x**3-0.5*op.domain.summands[1].x)
drift = op.domain.summands[1].pts2coeff(drift_pts)

shift = 5
u_0_pts = np.zeros_like(op.domain.summands[0].global_quad_pts)
u_0_pts[len(op.domain.summands[0].global_quad_pts) // 2 +shift] = 1.0  # Initial condition: delta function at the center
u_0 = op.domain.summands[0].get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(op.domain.summands[0].global_quad_pts)  # Example function to project
ones = op.domain.summands[0].get_fem_from_pts(ones_quad)
u_0 /= ones.T @ op.domain.summands[0].M @ u_0
print(op.domain.summands)
print(u_0 in op.domain.summands[0], drift in op.domain.summands[1])
u_final_fem = op(op.domain.join(u_0,drift))
u_final = op.domain.summands[0].DOF2pts @ u_final_fem

u_final[u_final<0] = 0
plt.plot(op.domain.summands[0].global_quad_pts,u_final)

In [ ]:
x = op.domain.rand()
_, deriv = op.linearize(x)
test_linearity(deriv)
test_linearity(deriv.adjoint)
test_adjoint(deriv)

In [ ]:
exact_data = u_final
x_plot = op.domain.summands[0].global_quad_pts
plt.plot(x_plot,exact_data, label= "exact data")

n_sample = 100
sample = np.random.poisson(lam = exact_data, size=(n_sample,u_final.size)).sum(axis = 0)/n_sample
plt.plot(x_plot,sample, label = "noisy data")

In [ ]:
from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers import RegularizationSetting
# from regpy.hilbert import L2

setting = RegularizationSetting(
    op = op,
    penalty = L2,
    data_fid = L2
)

solver = IrgnmCG(setting=setting,data = op.codomain.get_fem_from_pts(sample),regpar=0.3)


In [ ]:
one  = op.domain.ones()

print(setting.h_domain.vecsp.summands)
print([s.gram.codomain for s in setting.h_domain.summands])
print(setting.h_domain.norm(one))

In [ ]:
from regpy.stoprules import CountIterations

stop = CountIterations(10)

x,y = solver.run(stop)